# K-ABENA — DL Données Déséquilibrées : PyTorch ↔ TensorFlow

K-ABENA agit comme un **rééquilibrage implicite** : les images/textes de la classe majoritaire
bien appris ont une perte ≤ K (mineures) et sont candidats à l'exclusion,
tandis que la minorité mal apprise reste toujours active.

**Ratio simulé : 90% classe 0 / 10% classe 1**

| Méthode | Approche |
|---------|----------|
| Standard | Toutes observations — biais vers la majorité |
| Class weights | Pondération explicite par classe |
| **K-ABENA** | Filtrage implicite — les mineures (bien apprises) sont exclues |

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report

# Dataset déséquilibré 90/10
X, y = make_classification(
    n_samples=2000, n_features=20, weights=[0.90, 0.10],
    n_informative=10, random_state=42
)
X = StandardScaler().fit_transform(X).astype(np.float32)
y = y.astype(np.int64)

split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"Train — classe 0: {(y_train==0).sum()}, classe 1: {(y_train==1).sum()}")
print(f"Ratio : {(y_train==0).mean()*100:.0f}% / {(y_train==1).mean()*100:.0f}%")

K_ABENA = 0.20
N_ABENA = 0.30
EPOCHS  = 100

---
# VERSION PYTORCH — 3 approches comparées

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from kabena_ml.integrations.torch_utils import kabena_filter_torch

def build_mlp_pt(in_dim=20):
    return nn.Sequential(
        nn.Linear(in_dim, 64), nn.ReLU(),
        nn.Linear(64, 32), nn.ReLU(),
        nn.Linear(32, 2)
    )

X_pt = torch.tensor(X_train)
y_pt = torch.tensor(y_train)

# ── Approche 1 : Standard ─────────────────────────────────────────────────
m_std = build_mlp_pt(); opt = torch.optim.Adam(m_std.parameters(), 1e-3)
for _ in range(EPOCHS):
    loss = F.cross_entropy(m_std(X_pt), y_pt)
    opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad():
    p_std = m_std(torch.tensor(X_test)).argmax(1).numpy()
f1_std = f1_score(y_test, p_std, average='macro')
print(f"[PT] Standard              F1-macro: {f1_std:.4f}")

# ── Approche 2 : Class weights ────────────────────────────────────────────
n0, n1 = (y_train==0).sum(), (y_train==1).sum()
weights = torch.tensor([1/n0, 1/n1], dtype=torch.float32)
weights = weights / weights.sum() * 2
m_cw = build_mlp_pt(); opt = torch.optim.Adam(m_cw.parameters(), 1e-3)
for _ in range(EPOCHS):
    loss = F.cross_entropy(m_cw(X_pt), y_pt, weight=weights)
    opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad():
    p_cw = m_cw(torch.tensor(X_test)).argmax(1).numpy()
f1_cw = f1_score(y_test, p_cw, average='macro')
print(f"[PT] Class weights         F1-macro: {f1_cw:.4f}")

# ── Approche 3 : K-ABENA (+2 lignes vs standard) ─────────────────────────
m_ka = build_mlp_pt(); opt = torch.optim.Adam(m_ka.parameters(), 1e-3)
gains = []
for _ in range(EPOCHS):
    losses = F.cross_entropy(m_ka(X_pt), y_pt, reduction='none')  # ← 'none'
    mask   = kabena_filter_torch(losses, K=K_ABENA, N=N_ABENA)    # ← +1 ligne
    if not mask.any(): continue
    L_KA = losses[mask].mean()
    opt.zero_grad(); L_KA.backward(); opt.step()
    gains.append((1 - mask.sum().item()/len(y_pt))*100)
with torch.no_grad():
    p_ka = m_ka(torch.tensor(X_test)).argmax(1).numpy()
f1_ka = f1_score(y_test, p_ka, average='macro')
print(f"[PT] K-ABENA (K={K_ABENA}, N={N_ABENA})  F1-macro: {f1_ka:.4f} | gain={np.mean(gains):.1f}%")
print(f"\n{'─'*50}")
print("Rapport K-ABENA PyTorch :")
print(classification_report(y_test, p_ka, target_names=['Majoritaire', 'Minoritaire']))

---
# VERSION TENSORFLOW — 3 approches comparées

In [ ]:
import tensorflow as tf
from kabena_ml.integrations.tf_utils import KabenaCallback, KabenaTFTrainer
from kabena_ml import KabenaConfig

def build_mlp_tf():
    return tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=(20,)),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(2),
    ])

# ── Approche 1 : Standard ─────────────────────────────────────────────────
m_std_tf = build_mlp_tf()
m_std_tf.compile(optimizer='adam',
                 loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                 metrics=['accuracy'])
m_std_tf.fit(X_train, y_train, epochs=EPOCHS, batch_size=len(X_train), verbose=0)
p_std_tf = tf.argmax(m_std_tf(X_test, training=False), axis=1).numpy()
f1_std_tf = f1_score(y_test, p_std_tf, average='macro')
print(f"[TF] Standard              F1-macro: {f1_std_tf:.4f}")

# ── Approche 2 : Class weights ────────────────────────────────────────────
class_weight = {0: 1/n0 * len(y_train)/2, 1: 1/n1 * len(y_train)/2}
m_cw_tf = build_mlp_tf()
m_cw_tf.compile(optimizer='adam',
                loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                metrics=['accuracy'])
m_cw_tf.fit(X_train, y_train, epochs=EPOCHS, batch_size=len(X_train),
            class_weight=class_weight, verbose=0)
p_cw_tf = tf.argmax(m_cw_tf(X_test, training=False), axis=1).numpy()
f1_cw_tf = f1_score(y_test, p_cw_tf, average='macro')
print(f"[TF] Class weights         F1-macro: {f1_cw_tf:.4f}")

# ── Approche 3 : K-ABENA (+1 callback vs standard) ───────────────────────
m_ka_tf = build_mlp_tf()
m_ka_tf.compile(optimizer='adam',
                loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                metrics=['accuracy'])
ka_cb = KabenaCallback(K=K_ABENA, N=N_ABENA, verbose=False)
# AVANT : m_ka_tf.fit(X_train, y_train, ...)
# APRÈS :
m_ka_tf.fit(X_train, y_train, epochs=EPOCHS, batch_size=len(X_train),
            callbacks=[ka_cb], verbose=0)  # ← +1 callback
p_ka_tf = tf.argmax(m_ka_tf(X_test, training=False), axis=1).numpy()
f1_ka_tf = f1_score(y_test, p_ka_tf, average='macro')
print(f"[TF] K-ABENA (K={K_ABENA}, N={N_ABENA})  F1-macro: {f1_ka_tf:.4f} | {ka_cb.stats_summary()}")
print(f"\n{'─'*50}")
print("Rapport K-ABENA TensorFlow :")
print(classification_report(y_test, p_ka_tf, target_names=['Majoritaire', 'Minoritaire']))

## Récapitulatif

| Méthode | PyTorch | TensorFlow | Lignes ajoutées |
|---------|---------|------------|------------------|
| Standard | `loss = F.ce(...)` | `model.fit(...)` | 0 |
| Class weights | `F.ce(..., weight=w)` | `model.fit(..., class_weight=w)` | +1 |
| **K-ABENA** | `reduction='none'` + `mask = kabena_filter_torch(...)` | `callbacks=[KabenaCallback(K)]` | **+2 / +1** |

**Avantage K-ABENA sur class_weights** : pas besoin de connaître à l'avance les ratios de classes.
Le rééquilibrage est **adaptatif** — il évolue au fil de l'entraînement au fur et à mesure que
la classe majoritaire est mieux apprise.